In [1]:
# Setup: import libraries and frame the data from bronze layer table

from pyspark.sql.types import DateType
from pyspark.sql import functions as F 
from pyspark.sql import SparkSession


silver_delays = spark.read.format("delta").load(
    "abfss://EAA_WorkSpace@onelake.dfs.fabric.microsoft.com/"
    "EAA_LakeHouse.Lakehouse/Tables/bronze/apt_dly"
)


StatementMeta(, 0c9dad54-c0a0-41b4-89f7-04283f17ca7e, 3, Finished, Available, Finished, False)

In [2]:
# Map new and more descreptive names for the columns containing industry codes

column_renames = {
    "FLT_ARR_1": "TOTAL_ARRIVALS",

    "DLY_APT_ARR_1": "TOTAL_ATFM_DELAY_MINUTES",

    "DLY_APT_ARR_A_1": "ACCIDENT_INCIDENT_DELAY_MINUTES",

    "DLY_APT_ARR_C_1": "ATC_CAPACITY_DELAY_MINUTES",

    "DLY_APT_ARR_D_1": "DEICING_DELAY_MINUTES",

    "DLY_APT_ARR_E_1": "NON_ATC_EQUIPMENT_DELAY_MINUTES",

    "DLY_APT_ARR_G_1": "AERODROME_CAPACITY_DELAY_MINUTES",

    "DLY_APT_ARR_I_1": "ATC_INDUSTRIAL_ACTION_DELAY_MINUTES",

    "DLY_APT_ARR_M_1": "AIRSPACE_MANAGEMENT_DELAY_MINUTES",

    "DLY_APT_ARR_N_1": "NON_ATC_INDUSTRIAL_ACTION_DELAY_MINUTES",

    "DLY_APT_ARR_O_1": "OTHER_DELAY_MINUTES",

    "DLY_APT_ARR_P_1": "SPECIAL_EVENT_DELAY_MINUTES",

    "DLY_APT_ARR_R_1": "ATC_ROUTEING_DELAY_MINUTES",

    "DLY_APT_ARR_S_1": "ATC_STAFFING_DELAY_MINUTES",

    "DLY_APT_ARR_T_1": "ATC_EQUIPMENT_DELAY_MINUTES",

    "DLY_APT_ARR_V_1": "ENVIRONMENTAL_ISSUES_DELAY_MINUTES",

    "DLY_APT_ARR_W_1": "WEATHER_DELAY_MINUTES",

    "DLY_APT_ARR_NA_1": "NOT_SPECIFIED_DELAY_MINUTES",

    "FLT_ARR_1_DLY": "ATFM_DELAYED_ARRIVALS",

    "FLT_ARR_1_DLY_15": "ATFM_DELAYED_ARRIVALS_OVER_15_MINUTES",
}


StatementMeta(, 0c9dad54-c0a0-41b4-89f7-04283f17ca7e, 4, Finished, Available, Finished, False)

In [3]:
# Quality check: Check that all the necessary columns are available and no duplicates were created

required_source_columns = {
    "YEAR",
    "MONTH_NUM",
    "MONTH_MON",
    "FLT_DATE",
    "APT_ICAO",
    "APT_NAME",
    *column_renames.keys(),
}

missing_columns = required_source_columns - set(silver_delays.columns)

if missing_columns:
    raise ValueError(
        f"Missing required source columns: {sorted(missing_columns)}"
    )

if len(column_renames.values()) != len(set(column_renames.values())):
    raise ValueError("column_renames contains duplicate target names")


StatementMeta(, 0c9dad54-c0a0-41b4-89f7-04283f17ca7e, 5, Finished, Available, Finished, False)

In [4]:
integer_columns = {
    "YEAR",
    "MONTH_NUM",
    *column_renames.values(),
}

# Rename and cast in one transformation
silver_delays = silver_delays.select(
    *[
        (
            F.col(source_column).cast("int")
            if column_renames.get(source_column, source_column) in integer_columns
            else F.col(source_column)
        ).alias(column_renames.get(source_column, source_column))
        for source_column in silver_delays.columns
    ]
)

# Include the total and individual delay-minute columns
delay_minute_columns = [
    renamed_column
    for source_column, renamed_column in column_renames.items()
    if source_column.startswith("DLY_APT_ARR_")
]

silver_delays = silver_delays.fillna(
    0,
    subset=delay_minute_columns
)

# Clean up the remaining columns
silver_delays = (
    silver_delays
    .withColumn("MONTH_MON", F.trim("MONTH_MON"))
    .withColumn("FLT_DATE", F.col("FLT_DATE").cast("date"))
    .withColumn("APT_ICAO", F.trim("APT_ICAO"))
    .withColumn(
        "APT_NAME",
        F.regexp_replace(F.trim("APT_NAME"), r"\s*/\s*", "/")
    )
)

StatementMeta(, 0c9dad54-c0a0-41b4-89f7-04283f17ca7e, 6, Finished, Available, Finished, False)

In [5]:
# Add a string column for filtering by delay reason

delay_reason_columns = [
    column_name
    for column_name in column_renames.values()
    if column_name.endswith("_DELAY_MINUTES")
    and column_name != "TOTAL_ATFM_DELAY_MINUTES"
]

silver_delays = silver_delays.withColumn(
    "DELAY_REASONS_OVERVIEW",
    F.concat_ws(
        ", ",
        F.array_compact(
            F.array(
                *[
                    F.when(
                        F.coalesce(F.col(column_name), F.lit(0)) > 0,
                        F.lit(
                            column_name.removesuffix("_DELAY_MINUTES")
                        )
                    )
                    for column_name in delay_reason_columns
                ]
            )
        )
    )
)

# Add a standardized date column in preparation for the gold level

silver_delays = silver_delays.withColumn(
    "year_month",
    F.date_format(F.col("FLT_DATE"), "yyyyMM").cast("int")
)

StatementMeta(, 0c9dad54-c0a0-41b4-89f7-04283f17ca7e, 7, Finished, Available, Finished, False)

In [6]:
quality_rules = [
    (
        F.col("YEAR").isNull(),
        "YEAR_IS_NULL",
    ),
    (
        F.col("MONTH_NUM").isNull(),
        "MONTH_NUM_IS_NULL",
    ),
    (
        F.col("FLT_DATE").isNull(),
        "FLT_DATE_IS_NULL",
    ),
    (
        F.col("APT_ICAO").isNull()
        | (F.length(F.trim(F.col("APT_ICAO"))) == 0),
        "APT_ICAO_IS_MISSING",
    ),
    (
        F.col("MONTH_NUM").isNotNull()
        & ~F.col("MONTH_NUM").between(1, 12),
        "INVALID_MONTH_NUM",
    ),
    (
        F.col("APT_ICAO").isNotNull()
        & ~F.col("APT_ICAO").rlike(r"^[A-Z]{4}$"),
        "INVALID_APT_ICAO",
    ),
    (
        F.col("TOTAL_ARRIVALS").isNull(),
        "TOTAL_ARRIVALS_IS_NULL",
    ),
    (
        F.col("TOTAL_ATFM_DELAY_MINUTES").isNull(),
        "TOTAL_ATFM_DELAY_MINUTES_IS_NULL",
    ),
    (
        F.col("ATFM_DELAYED_ARRIVALS_OVER_15_MINUTES").isNotNull()
        & F.col("ATFM_DELAYED_ARRIVALS").isNotNull()
        & (
            F.col("ATFM_DELAYED_ARRIVALS_OVER_15_MINUTES")
            > F.col("ATFM_DELAYED_ARRIVALS")
        ),
        "OVER_15_MINUTES_EXCEEDS_DELAYED_ARRIVALS",
    ),
    (
        F.col("FLT_DATE").isNotNull()
        & F.col("YEAR").isNotNull()
        & (
            F.year(F.col("FLT_DATE"))
            != F.col("YEAR")
        ),
        "DATE_YEAR_MISMATCH",
    ),
    (
        F.col("FLT_DATE").isNotNull()
        & F.col("MONTH_NUM").isNotNull()
        & (
            F.month(F.col("FLT_DATE"))
            != F.col("MONTH_NUM")
        ),
        "DATE_MONTH_MISMATCH",
    ),
]


# Add non-negative checks for all renamed numeric columns.

for column_name in column_renames.values():
    quality_rules.append(
        (
            F.col(column_name).isNotNull()
            & (F.col(column_name) < 0),
            f"NEGATIVE_{column_name}",
        )
    )


# Evaluate all quality rules and add the rejection reasons to each row.
# Multiple rejection reasons are stored as a comma-separated string.

checked_delays = silver_delays.withColumn(
    "REJECTION_REASON",
    F.concat_ws(
        ", ",
        F.array_compact(
            F.array(
                *[
                    F.when(condition, F.lit(error_name))
                    for condition, error_name in quality_rules
                ]
            )
        ),
    ),
)


# Gather all rows that failed at least one quality rule.
# Retain REJECTION_REASON and record when each row was rejected.

rejected_rows = (
    checked_delays
    .filter(F.length(F.col("REJECTION_REASON")) > 0)
    .withColumn("REJECTED_AT", F.current_timestamp())
)


# Keep rows that passed every quality rule.
# Remove the empty rejection-reason column from accepted rows.

accepted_rows = (
    checked_delays
    .filter(F.length(F.col("REJECTION_REASON")) == 0)
    .drop("REJECTION_REASON")
)


# Report the result without stopping the pipeline.

rejected_row_count = rejected_rows.count()

print(
    f"Data-quality checks completed. "
    f"{rejected_row_count} rows were added to rejected_rows."
)

StatementMeta(, 0c9dad54-c0a0-41b4-89f7-04283f17ca7e, 8, Finished, Available, Finished, False)

Data-quality checks completed. 0 rows were added to rejected_rows.


In [7]:
# Check the dataframe

display(rejected_rows)

display(silver_delays)

StatementMeta(, 0c9dad54-c0a0-41b4-89f7-04283f17ca7e, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b664ba1e-e668-482b-9db3-53c53a388de1)

SynapseWidget(Synapse.DataFrame, 2b04c766-f505-49f0-9092-f0e934a5b2a7)

In [9]:
# Write the dataframes into a table

(
    silver_delays.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.atfm_delays")
)


rejected_rows.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.atfm_delays_rejected")

StatementMeta(, 0c9dad54-c0a0-41b4-89f7-04283f17ca7e, 11, Finished, Available, Finished, False)

In [10]:
# Check row counts for original table and the modified table

row_count_bronze = spark.table("bronze.apt_dly").count()

row_count_silver = spark.table("silver.atfm_delays").count()

print(row_count_bronze)
print(row_count_silver)

StatementMeta(, 0c9dad54-c0a0-41b4-89f7-04283f17ca7e, 12, Finished, Available, Finished, False)

593794
593794
